# Asia C Failure Diagnosis

Postmortem of Universe C (HK/JP) locked pairs under the frozen trad-z / OLS / 1d stack.
Not a new hypothesis — does not change `KEEP_PAIR_IDS` or the star stack.

**Pairs:** `1398.HK|0939.HK`, `1288.HK|3328.HK`, `8306.T|8316.T`  
**Default window:** research IS only (`date <= RESEARCH_IS_END`). Set `USE_OUT_OF_SAMPLE_DATA = True` to include sealed OOS.  
**Helpers:** `04_backtest/s2_coint/diagnosis.py`

## 0. Imports & Config

In [ ]:
from __future__ import annotations

import os
import sys

import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.diagnosis import (
    check_fill_timing,
    failure_scorecard,
    plotly_pair_diagnosis,
    print_extreme_trades,
    simulate_and_enrich_panel,
    slice_panel_window,
)
from backtest.s2_coint.research import (
    FROZEN_C_PAIRS,
    RESEARCH_IS_END_C,
    load_universe_c_panels,
    panel_paths,
)
from strategies.s2_coint.baseline import ENTRY_Z, EXIT_Z

USE_OUT_OF_SAMPLE_DATA = False
Z_WINDOW = 60
RESEARCH_IS_END = RESEARCH_IS_END_C
PAIRS = list(FROZEN_C_PAIRS)
N_EXTREME = 3

print("ROOT", ROOT)
print("PAIRS", PAIRS)
print("RESEARCH_IS_END", RESEARCH_IS_END)
print("USE_OUT_OF_SAMPLE_DATA", USE_OUT_OF_SAMPLE_DATA)
print("ENTRY_Z", ENTRY_Z, "EXIT_Z", EXIT_Z, "Z_WINDOW", Z_WINDOW)

## 1. Failure-mode legend

| Section | Failure looks like |
|---------|-------------------|
| **Opportunity** | Few entries / round-trips; `% days \|z\| > ENTRY_Z` near 0 (flat z, almost never crosses bands). |
| **Costs** | `ann_sharpe_gross` clearly positive while `ann_sharpe_net` ≤ 0, or `cost_bps_year` large vs edge. |
| **No edge** | Both gross and net Sharpe ≤ 0 (or near 0 with noisy PnL). |
| **Noise / horizon** | Many trades, median hold ≪ median half-life, expectancy near zero. |
| **Coint health** | High median/last ADF p, low `% ADF < 0.05`, or large `beta_std` (unstable hedge). |
| **Fill timing** | Any `all_ok == False` in the timing table (same-bar close fill or missing open). |

**Gross vs net:** Gross Sharpe ignores commissions/spread/slippage; net includes them — if gross ≫ 0 and net ≤ 0, costs ate the edge; if both ≤ 0, there was no edge even before costs.

## 2. Load panels

Requires `s2_panel_C_1d_train.parquet` (and `s2_panel_C_1d_full.parquet` when `USE_OUT_OF_SAMPLE_DATA` is True).
Rebuild: run `H-001_universes.ipynb` lock step, then `01_data/data_files/s2_coint/s2_pair_panel.ipynb`.

In [ ]:
train_path, full_path = panel_paths("1d", root=ROOT)
print("train", train_path, "exists", os.path.isfile(train_path))
print("full ", full_path, "exists", os.path.isfile(full_path))

train, full = load_universe_c_panels("1d", PAIRS, root=ROOT)
source = full if USE_OUT_OF_SAMPLE_DATA else train
panel = slice_panel_window(source, RESEARCH_IS_END, USE_OUT_OF_SAMPLE_DATA)
panel = panel.loc[panel["pair_id"].isin(PAIRS)].copy()

print("rows", len(panel), "pairs", sorted(panel["pair_id"].unique().tolist()))
print("date range", panel["date"].min(), "→", panel["date"].max())
if USE_OUT_OF_SAMPLE_DATA:
    n_oos = int((panel["date"] > pd.Timestamp(RESEARCH_IS_END)).sum())
    print("OOS rows", n_oos)

## 3. Failure scorecard

Per-pair opportunity, net/gross edge, cost drag, and cointegration health on the loaded window.

In [ ]:
scorecard = failure_scorecard(
    panel,
    entry_z=ENTRY_Z,
    exit_z=EXIT_Z,
    gross=True,
)
scorecard

### 3.1 Optional IS vs OOS split

Runs only when `USE_OUT_OF_SAMPLE_DATA` is True.

In [ ]:
if USE_OUT_OF_SAMPLE_DATA:
    is_panel = panel.loc[panel["date"] <= pd.Timestamp(RESEARCH_IS_END)].copy()
    oos_panel = panel.loc[panel["date"] > pd.Timestamp(RESEARCH_IS_END)].copy()
    print("=== Research IS ===")
    display(failure_scorecard(is_panel, entry_z=ENTRY_Z, exit_z=EXIT_Z, gross=True))
    print("=== Sealed OOS ===")
    display(failure_scorecard(oos_panel, entry_z=ENTRY_Z, exit_z=EXIT_Z, gross=True))
else:
    print("USE_OUT_OF_SAMPLE_DATA=False — skip IS/OOS split tables.")

## 4. Extreme trades

Best/worst round-trips by **net** `pnl_pct` (percent of that pair’s capital, costs included). Use the printed dates to zoom the Plotly charts.

In [ ]:
trades_enriched, returns_by_pair = simulate_and_enrich_panel(
    panel, entry_z=ENTRY_Z, exit_z=EXIT_Z
)
print("n round-trips", len(trades_enriched))
print_extreme_trades(trades_enriched, n=N_EXTREME)
trades_enriched.sort_values("pnl_pct", ascending=False).head(10)

## 5. Interactive plots

Per pair: spread with Bollinger bands at `± ENTRY_Z · σ` over `Z_WINDOW` (same standardization as trad-z), green/red entry/exit markers, rolling ADF p, beta, cumulative net PnL.
When `USE_OUT_OF_SAMPLE_DATA` is True, a vertical line marks `RESEARCH_IS_END`.

In [ ]:
figures = []
for pair_id in PAIRS:
    g = panel.loc[panel["pair_id"] == pair_id].copy()
    t = trades_enriched.loc[trades_enriched["pair_id"] == pair_id].copy()
    fig = plotly_pair_diagnosis(
        g,
        t,
        entry_z=ENTRY_Z,
        z_window=Z_WINDOW,
        pair_returns=returns_by_pair.get(pair_id),
        is_end=RESEARCH_IS_END if USE_OUT_OF_SAMPLE_DATA else None,
        title=f"{pair_id} · {'IS+OOS' if USE_OUT_OF_SAMPLE_DATA else 'research IS'}",
    )
    figures.append(fig)
    fig.show()
print("plotted", len(figures), "pairs")

## 6. Fill-timing check

**Pass:** decision uses finite z on close of signal bar `t`; fill is open of the next panel session (`entry_date`); signal date ≠ fill date (not same-bar close fill).

**Fail:** any `all_ok == False` — inspect that trade’s `signal_date` / `entry_date` / open columns.

In [ ]:
timing = check_fill_timing(trades_enriched, panel)
n_fail = int((~timing["all_ok"]).sum()) if not timing.empty else 0
print("timing rows", len(timing), "failures", n_fail)
timing